<a href="https://colab.research.google.com/github/Joell-Alex1/Chess-Comment-Toxicity-Domain-Classification/blob/main/Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import string
import re   # you forgot this import

sentences = {
    "toxic": [
        "You are using     Stockfish!    ",
        "Nice !!//?game bro!! running"
    ]
}

df = pd.DataFrame(sentences)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
    text = text.translate(translator)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)


    return text

# Apply function to each row
df["cleaned"] = df["toxic"].apply(clean_text)

print(df)




In [ ]:
!pip install nltk

In [ ]:
import nltk
nltk.download("punkt")
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize


df["tokens"] = df["cleaned"].apply(word_tokenize)

print(df["tokens"])

Remove the STOPWORDS


In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words("english"))
def remove_stopwords(tokens):

  filtered=[]
  for words in tokens:
    if words not in stop_words:
      filtered.append(words)
  return filtered


In [ ]:
# lambda parameter: expression
# list comp is expression iteration if/else
df["tokens_no_stop"] = df["tokens"].apply(
    lambda words: [word for word in words if word not in stop_words]
)
print(df["tokens_no_stop"])

In [ ]:
nltk.download("wordnet")

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df["tokens_lemma"] = df["tokens_no_stop"].apply(
    lambda words: [lemmatizer.lemmatize(word, pos="v") for word in words]
)
df["final_text"] = df["tokens_lemma"].apply(lambda words: " ".join(words))
print(df)

In [ ]:
!pip install youtube-comment-downloader

In [ ]:
from itertools import islice
from youtube_comment_downloader import *
downloader = YoutubeCommentDownloader()
comments_list=[]
comments = downloader.get_comments_from_url('https://www.youtube.com/watch?v=aDUmS_MJceU', sort_by=SORT_BY_POPULAR)
for comment in islice(comments, 100):

    print(clean_text(comment['text']))
    comments_list.append(clean_text(comment['text']))


df = pd.DataFrame(comments_list, columns=["toxic"])

# Run full pipeline on scraped data
df["cleaned"] = df["toxic"].apply(clean_text)
df["tokens"] = df["cleaned"].apply(word_tokenize)
df["tokens_no_stop"] = df["tokens"].apply(remove_stopwords)
df["tokens_lemma"] = df["tokens_no_stop"].apply(
    lambda words: [lemmatizer.lemmatize(word, pos="v") for word in words]
)
df["final_text"] = df["tokens_lemma"].apply(lambda words: " ".join(words))
print(df)

In [ ]:
!pip install detoxify

In [ ]:
from detoxify import Detoxify
results = Detoxify('original').predict(df["toxic"].tolist())

In [ ]:
print(df)

In [ ]:
df["tokens_no_stop"] = df["toxic"].apply(remove_stopwords)

In [ ]:
df["toxicity"] = results["toxicity"]
df["severe_toxicity"] = results["severe_toxicity"]
df["obscene"] = results["obscene"]
df["threat"] = results["threat"]
df["insult"] = results["insult"]
df["identity_attack"] = results["identity_attack"]
print(df)



In [ ]:

chess_terms = {"engine", "stockfish", "elo", "niemann", "hans", "magnus", "carlsen", "prep", "gm"}

accusation_terms = {"using", "cheater", "caught", "banned", "reported", "sus", "suspicious",
                    "smurf", "sandbag", "cheat", "cheats", "leaked", "assistance", "traitor", "bot"}

direct_toxic = {"ez", "ezez", "patzer", "noob", "trash", "donkey", "woodpusher", "duffer", "trashcan", "zzz"}
def classify_comment(comment):
    has_chess_term = any(word in comment for word in chess_terms)
    has_accusation = any(word in comment for word in accusation_terms)
    toxic = any(re.search(r"\b" + word + r"\b", comment) for word in direct_toxic)

    if toxic:
        return "direct_toxic"
    if has_chess_term and has_accusation:
        return "cheating_accusation"
    elif has_chess_term:
        return "chess_mention"
    else:
        return "normal"

df["chess_flag"] = df["final_text"].apply(classify_comment)
print(df["chess_flag"].value_counts())

print(df)

In [ ]:
print(df["chess_flag"].value_counts())


In [ ]:
# from sklearn.metrics import classification_report

# labeled_df = pd.read_csv("chess_comments_labeled.csv")
# print(classification_report(labeled_df["manual_label"], df["chess_flag"]))

In [ ]:
# # Detoxify flags anything above 0.5 as toxic
# labeled_df["detoxify_flag"] = labeled_df["toxicity"].apply(
#     lambda x: "cheating_accusation" if x > 0.5 else "normal"
# )
# print(classification_report(labeled_df["manual_label"], labeled_df["detoxify_flag"]))

In [ ]:
df.to_csv("chess_comments.csv", index=False)